In [ ]:
## Time-dependent PRCC for DDE Brood Model

import numpy as np
from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde
from scipy.stats import qmc, spearmanr
import matplotlib.pyplot as plt

# Helpers
def pos(x):
    return (x + Abs(x)) / 2

def PRCC(X, Y):
    p = X.shape[1]
    prcc = np.zeros(p)
    for i in range(p):
        Xi = X[:, i]
        Xr = np.delete(X, i, axis=1)

        r_xy, _ = spearmanr(Xi, Y)
        r_xr = np.array([spearmanr(Xi, Xr[:, j])[0] for j in range(p-1)])
        r_yr = np.array([spearmanr(Y,  Xr[:, j])[0] for j in range(p-1)])

        prcc[i] = (r_xy - np.dot(r_xr, r_yr)) / \
                  np.sqrt((1 - np.dot(r_xr, r_xr)) * (1 - np.dot(r_yr, r_yr)))
    return prcc

# MODEL
def run_model_dde_timeseries(params, times):
    
    # Fixed parameters
    phi_B   = 0.018
    phi_H   = 0.007
    b       = 500
    a       = 500
    k       = 5000
    kappa   = 1/9
    tau     = 12
    gamma_B = 0.003
    theta   = 105

    # Varied parameters
    chi, sigma, epsilon_B, gamma_H, delta_1, delta_2, epsilon_T, gamma_T = params

    # Initial conditions
    B0, H0, T0, V0, f0 = 5000, 10000, 2, 1, 1000
    q0 = delta_1*T0/(a+B0) + delta_2*V0/(a+B0)
    I0 = q0 * tau

    # Symbolic states
    B, H, T, V, f, I = y(0), y(1), y(2), y(3), y(4), y(5)
    B_tau, T_tau, V_tau = y(0, t-tau), y(2, t-tau), y(3, t-tau)

    # Nonnegativity
    Bp, Hp, Tp, Vp, fp, Ip = map(pos, (B,H,T,V,f,I))
    Btp, Ttp, Vtp = map(pos, (B_tau,T_tau,V_tau))

    # Model terms
    S = (fp/(b+fp)) * (Hp/(k+Hp))
    infestation = delta_1 * (Bp/(a+Bp)) * Tp
    infection   = delta_2 * (Bp/(a+Bp)) * Vp
    q_now = delta_1*Tp/(a+Bp) + delta_2*Vp/(a+Bp)
    q_tau = delta_1*Ttp/(a+Btp) + delta_2*Vtp/(a+Btp)
    dI = q_now - q_tau
    exp_B = exp(-gamma_B*tau - Ip)
    Phi = chi + sigma*cos(2*pi*(t-theta)/365)

    # DERIVATIVES
    dB = epsilon_B*S - infestation - infection - gamma_B*Bp - kappa*exp_B*Btp
    dH = kappa*exp_B*Btp - gamma_H*Hp
    dT = epsilon_T*infestation - gamma_T*Tp
    dV = epsilon_T*infection - gamma_T*Vp
    df = Phi*(Hp/(k+Hp)) - phi_B*Bp - phi_H*Hp

    # Stabilization
    lam = 120
    dB += lam*(Bp-B); dH += lam*(Hp-H)
    dT += lam*(Tp-T); dV += lam*(Vp-V)
    df += lam*(fp-f); dI += lam*(Ip-I)

    # Setup DDE Solver
    DDE = jitcdde([dB,dH,dT,dV,df,dI])
    DDE.past_from_function(lambda s: [B0,H0,T0,V0,f0,I0])
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(max_step=0.02)

    # Integrate over times
    B_series = np.zeros(len(times))
    for i, tt in enumerate(times):
        B_series[i] = DDE.integrate(tt)[0]   # B(t)
    return B_series

In [ ]:
N = 200  # number of LHS samples
param_names = [r"$\chi$", r"$\sigma$", r"$\epsilon_B$", r"$\gamma_H$",
               r"$\delta_1$", r"$\delta_2$", r"$\epsilon_T$", r"$\gamma_T$"]

bounds = np.array([
    [500, 700],      # chi
    [200, 400],      # sigma
    [1500, 2000],    # epsilon_B
    [0.0, 0.17],     # gamma_H
    [0.0, 0.1],      # delta_1
    [0.0, 0.1],      # delta_2
    [1, 2.1],        # epsilon_T
    [1/28, 1/12],    # gamma_T
])

# LHS sampling
sampler = qmc.LatinHypercube(d=bounds.shape[0])
X = qmc.scale(sampler.random(N), bounds[:, 0], bounds[:, 1])

# Times settings
times = np.arange(0, 366, 1)  # 0..365 daily
Y_time = np.zeros((N, len(times)))

# Run ensemble
print("Running ensemble...")
for i in range(N):
    Y_time[i, :] = run_model_dde_timeseries(X[i], times)  # returns B(t)
    if (i + 1) % 20 == 0:
        print(f"Completed {i+1}/{N}")

# COMPUTE PRCC
PRCC_time = np.full((len(times), X.shape[1]), np.nan)

for k in range(len(times)):
    yk = Y_time[:, k]

    # degenerate output -> leave NaN for now
    if np.std(yk, ddof=1) < 1e-10:
        continue

    r = PRCC(X, yk)
    PRCC_time[k, :] = np.clip(r, -1, 1)  # enforce valid PRCC bounds

# Fill NaN rows from nearest valid row (so all plotted times exist, incl. t=0)
valid_rows = np.where(np.all(np.isfinite(PRCC_time), axis=1))[0]
if len(valid_rows) > 0:
    for k in np.where(~np.all(np.isfinite(PRCC_time), axis=1))[0]:
        nearest = valid_rows[np.argmin(np.abs(valid_rows - k))]
        PRCC_time[k, :] = PRCC_time[nearest, :]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# PRCC_time: shape (n_times, n_params)
n_params = len(param_names)
n_times = len(times)

# Fixed 13 bars at t = 0, 30, ..., 360
time_days = np.arange(0, 361, 30)
time_indices = np.array([np.argmin(np.abs(times - d)) for d in time_days])
time_labels = [f"t={int(times[i])}" for i in time_indices]

# Values for plotting (already repaired in simulation block)
PRCC_vals = np.clip(PRCC_time[time_indices, :], -1, 1)

# Colors (one per time index)
colors = [
    (148/255, 203/255, 236/255),  # Light blue
    (204/255, 121/255, 167/255),  # Purple
    (255/255, 194/255, 10/255),   # Yellow
    (93/255, 58/255, 155/255),    # Violet
    (220/255, 50/255, 32/255),    # Red
    (0/255, 108/255, 209/255),    # Blue
    (0/255, 0/255, 0/255),        # Black
    (230/255, 97/255, 0/255),     # Orange
    (187/255, 187/255, 187/255),  # Gray
    (211/255, 95/255, 183/255),   # Pink
    (153/255, 79/255, 0/255),     # Brown
    (26/255, 255/255, 26/255),    # Green
    (254/255, 254/255, 98/255)    # Light Yellow
]

fig, ax = plt.subplots(figsize=(7, 5))

# Cluster geometry
n_show = len(time_indices)  # 13
group_gap = 1.25            # space between parameter groups
cluster_width = 1        # keeps bars reasonably thick
bar_width = cluster_width / n_show

x = np.arange(n_params) * group_gap
offsets = (np.arange(n_show) - (n_show - 1) / 2) * bar_width

# Plot clustered bars
for j in range(n_show):
    c = colors[j]
    ax.bar(
        x + offsets[j],
        PRCC_vals[j, :],
        width=bar_width,
        color=c,
        edgecolor=c,
        linewidth=0.4,
        zorder=3
    )

ax.set_xticks(x)
ax.set_xticklabels(param_names, fontsize=18)
ax.set_xlabel("Parameter", color='black', fontsize=15, labelpad=10, ha='center')
ax.set_ylabel("Sensitivity index", color='black', fontsize=15, labelpad=10, ha='center')
ax.set_ylim(-1, 1)
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
ax.tick_params(axis='y', which='major', labelsize=18, width=1.6, length=6, colors='black')
ax.set_title("PRCC for Brood Population", fontsize=15)
ax.tick_params(axis='x', which='major', labelsize=18, width=1.6, length=6, colors='black')
ax.axhline(0, color='black', linewidth=1, zorder=1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.text(-0.11, 1.15, "(c)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')

plt.tight_layout()
plt.savefig("Sensitivity_B2.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
## Time-dependent PRCC for DDE Hive Bee Model

import numpy as np
from symengine import exp, cos, pi, Abs
from jitcdde import y, t, jitcdde
from scipy.stats import qmc, spearmanr
import matplotlib.pyplot as plt

# Helper
def pos(x):
    return (x + Abs(x)) / 2

def PRCC(X, Y):
    p = X.shape[1]
    prcc = np.zeros(p)
    for i in range(p):
        Xi = X[:, i]
        Xr = np.delete(X, i, axis=1)

        r_xy, _ = spearmanr(Xi, Y)
        r_xr = np.array([spearmanr(Xi, Xr[:, j])[0] for j in range(p-1)])
        r_yr = np.array([spearmanr(Y,  Xr[:, j])[0] for j in range(p-1)])

        prcc[i] = (r_xy - np.dot(r_xr, r_yr)) / \
                  np.sqrt((1 - np.dot(r_xr, r_xr)) * (1 - np.dot(r_yr, r_yr)))
    return prcc

# MODEL
def run_model_dde_timeseries(params, times):
    
    # Fixed parameters
    phi_B   = 0.018
    phi_H   = 0.007
    b       = 500
    a       = 500
    k       = 5000
    kappa   = 1/9
    tau     = 12
    gamma_B = 0.003
    theta   = 105
    

    # Varied parameters
    chi, sigma, epsilon_B, gamma_H, delta_1, delta_2, epsilon_T, gamma_T = params

    # Initial conditions
    B0, H0, T0, V0, f0 = 5000, 10000, 2, 1, 1000
    q0 = delta_1*T0/(a+B0) + delta_2*V0/(a+B0)
    I0 = q0 * tau

    # Symbolic states
    B, H, T, V, f, I = y(0), y(1), y(2), y(3), y(4), y(5)
    B_tau, T_tau, V_tau = y(0, t-tau), y(2, t-tau), y(3, t-tau)

    # Nonnegativity
    Bp, Hp, Tp, Vp, fp, Ip = map(pos, (B,H,T,V,f,I))
    Btp, Ttp, Vtp = map(pos, (B_tau,T_tau,V_tau))

    # Model terms
    S = (fp/(b+fp)) * (Hp/(k+Hp))
    infestation = delta_1 * (Bp/(a+Bp)) * Tp
    infection   = delta_2 * (Bp/(a+Bp)) * Vp
    q_now = delta_1*Tp/(a+Bp) + delta_2*Vp/(a+Bp)
    q_tau = delta_1*Ttp/(a+Btp) + delta_2*Vtp/(a+Btp)
    dI = q_now - q_tau
    exp_B = exp(-gamma_B*tau - Ip)
    Phi = chi + sigma*cos(2*pi*(t-theta)/365)

    # DERIVATIVES
    dB = epsilon_B*S - infestation - infection - gamma_B*Bp - kappa*exp_B*Btp
    dH = kappa*exp_B*Btp - gamma_H*Hp
    dT = epsilon_T*infestation - gamma_T*Tp
    dV = epsilon_T*infection - gamma_T*Vp
    df = Phi*(Hp/(k+Hp)) - phi_B*Bp - phi_H*Hp

    # Stabilization
    lam = 120
    dB += lam*(Bp-B); dH += lam*(Hp-H)
    dT += lam*(Tp-T); dV += lam*(Vp-V)
    df += lam*(fp-f); dI += lam*(Ip-I)

    # Setup DDE Solver
    DDE = jitcdde([dB,dH,dT,dV,df,dI])
    DDE.past_from_function(lambda s: [B0,H0,T0,V0,f0,I0])
    DDE.step_on_discontinuities()
    DDE.set_integration_parameters(max_step=0.02)

    # Integrate over times
    H_series = np.zeros(len(times))
    for i, tt in enumerate(times):
        H_series[i] = DDE.integrate(tt)[1]   # H(t)
    return H_series

In [ ]:
N = 200  # number of LHS samples
param_names = [r"$\chi$", r"$\sigma$", r"$\epsilon_B$", r"$\gamma_H$",
               r"$\delta_1$", r"$\delta_2$", r"$\epsilon_T$", r"$\gamma_T$"]

bounds = np.array([
    [500, 700],      # chi
    [200, 400],      # sigma
    [1500, 2000],    # epsilon_B
    [0.0, 0.17],     # gamma_H
    [0.0, 0.1],      # delta_1
    [0.0, 0.1],      # delta_2
    [1, 2.1],        # epsilon_T
    [1/28, 1/12],    # gamma_T
])

# LHS sampling
sampler = qmc.LatinHypercube(d=bounds.shape[0])
X = qmc.scale(sampler.random(N), bounds[:, 0], bounds[:, 1])

# Times settings
times = np.arange(0, 366, 1)  # 0..365 daily
Y_time = np.zeros((N, len(times)))

# Run ensemble
print("Running ensemble...")
for i in range(N):
    Y_time[i, :] = run_model_dde_timeseries(X[i], times)  # returns B(t)
    if (i + 1) % 20 == 0:
        print(f"Completed {i+1}/{N}")

# COMPUTE PRCC
PRCC_time = np.full((len(times), X.shape[1]), np.nan)

for k in range(len(times)):
    yk = Y_time[:, k]

    # degenerate output -> leave NaN for now
    if np.std(yk, ddof=1) < 1e-10:
        continue

    r = PRCC(X, yk)
    PRCC_time[k, :] = np.clip(r, -1, 1)  # enforce valid PRCC bounds

# Fill NaN rows from nearest valid row (so all plotted times exist, incl. t=0)
valid_rows = np.where(np.all(np.isfinite(PRCC_time), axis=1))[0]
if len(valid_rows) > 0:
    for k in np.where(~np.all(np.isfinite(PRCC_time), axis=1))[0]:
        nearest = valid_rows[np.argmin(np.abs(valid_rows - k))]
        PRCC_time[k, :] = PRCC_time[nearest, :]

In [ ]:
# import matplotlib.pyplot as plt
#import numpy as np

# PRCC_time: shape (n_times, n_params)
n_params = len(param_names)
n_times = len(times)

# Fixed 13 bars at t = 0, 30, ..., 360
time_days = np.arange(0, 361, 30)
time_indices = np.array([np.argmin(np.abs(times - d)) for d in time_days])
time_labels = [f"t={int(times[i])}" for i in time_indices]

# Values for plotting (already repaired in simulation block)
PRCC_vals = np.clip(PRCC_time[time_indices, :], -1, 1)

# Colors for bars: each color corresponds to the time index (first bar, second bar, ...)
colors = [
    (148/255, 203/255, 236/255),  # Light blue 
    (204/255, 121/255, 167/255),  # purple 
    (255/255, 194/255, 10/255),   # Yellow 
    (93/255, 58/255, 155/255),    # Violet 
    (220/255, 50/255, 32/255),    # Red
    (0/255, 108/255, 209/255),    # Blue 
    (0/255, 0/255, 0/255),        # black 
    (230/255, 97/255, 0/255),     # Orange 
    (187/255, 187/255, 187/255),  # Gray 
    (211/255, 95/255, 183/255),   # Pink 
    (153/255, 79/255, 0/255),     # Brown
    (26/255, 255/255, 26/255),    # Green
    (254/255, 254/255, 98/255)    # Light Yellow 
]

fig, ax = plt.subplots(figsize=(7,5))

# Cluster geometry
n_show = len(time_indices)  # 13
group_gap = 1.25            # space between parameter groups
cluster_width = 1        # keeps bars reasonably thick
bar_width = cluster_width / n_show

x = np.arange(n_params) * group_gap
offsets = (np.arange(n_show) - (n_show - 1) / 2) * bar_width

# Plot clustered bars
for j in range(n_show):
    c = colors[j]
    ax.bar(
        x + offsets[j],
        PRCC_vals[j, :],
        width=bar_width,
        color=c,
        edgecolor=c,
        linewidth=0.4,
        zorder=3
    )

ax.set_xticks(x)
ax.set_xticklabels(param_names, fontsize=18)
ax.set_xlabel("Parameter", color='black', fontsize=15, labelpad=10, ha='center')
ax.set_ylabel("Sensitivity index", color='black', fontsize=15, labelpad=10, ha='center')
ax.set_ylim(-1, 1)
ax.set_yticks([-1, -0.5, 0, 0.5, 1])
ax.tick_params(axis='y', which='major', labelsize=18, width=1.6, length=6, colors='black')
ax.set_title("PRCC for Hive Bee Population", fontsize=15)
ax.tick_params(axis='x', which='major', labelsize=18, width=1.6, length=6, colors='black')
ax.axhline(0, color='black', linewidth=1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.text(-0.11, 1.15, "(d)", transform=ax.transAxes, color='black', fontsize=20, va='top', ha='left')

plt.tight_layout()
plt.savefig("Sensitivity_H2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
## Combine Pics
from PIL import Image, ImageOps
import numpy as np

def autocrop(im, pad=6):
    im = im.convert("RGB")
    arr = np.asarray(im).astype(np.int16)
    mask = np.any(arr < 245, axis=2)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return im
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(im.size[0], x1 + pad); y1 = min(im.size[1], y1 + pad)
    return im.crop((x0, y0, x1, y1))

def pad_to_height(im, target_h):
    w, h = im.size
    if h == target_h:
        return im
    top = (target_h - h)//2
    bottom = target_h - h - top
    return ImageOps.expand(im, border=(0, top, 0, bottom), fill="white")

# Load and crop 
imB = autocrop(Image.open("Sensitivity_B2.png"), pad=6)
imH = autocrop(Image.open("Sensitivity_H2.png"), pad=6)

# Match heights
target_h = max(imB.size[1], imH.size[1])
imB = pad_to_height(imB, target_h)
imH = pad_to_height(imH, target_h)

# Pure white space
gap = 40
space = Image.new("RGB", (gap, target_h), (255, 255, 255))  # WHITE

# Stitch
combined = Image.new("RGB", (imB.size[0] + gap + imH.size[0], target_h), (255, 255, 255))
combined.paste(imB, (0, 0))
combined.paste(space, (imB.size[0], 0))
combined.paste(imH, (imB.size[0] + gap, 0))

combined.save("Sensitivity_2.png", dpi=(600, 600))
print("Saved: Sensitivity_2.png")

In [ ]:
# Sensitivity 1 and 2
from PIL import Image, ImageOps
import numpy as np

def autocrop(im, pad=6):
    im = im.convert("RGB")
    arr = np.asarray(im).astype(np.int16)
    mask = np.any(arr < 245, axis=2)
    coords = np.argwhere(mask)
    if coords.size == 0:
        return im
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    x0 = max(0, x0 - pad); y0 = max(0, y0 - pad)
    x1 = min(im.size[0], x1 + pad); y1 = min(im.size[1], y1 + pad)
    return im.crop((x0, y0, x1, y1))

def pad_to_width(im, target_w):
    w, h = im.size
    if w == target_w:
        return im
    left = (target_w - w) // 2
    right = target_w - w - left
    return ImageOps.expand(im, border=(left, 0, right, 0), fill="white")

# Load and crop
im_apr = autocrop(Image.open("Sensitivity_1.png"), pad=6)
im_oct = autocrop(Image.open("Sensitivity_2.png"), pad=6)

# Match widths for vertical stacking
target_w = max(im_apr.size[0], im_oct.size[0])
im_apr = pad_to_width(im_apr, target_w)
im_oct = pad_to_width(im_oct, target_w)

# White gap between rows
gap = 40
space = Image.new("RGB", (target_w, gap), (255, 255, 255))

# Stitch vertically
combined_h = im_apr.size[1] + gap + im_oct.size[1]
combined = Image.new("RGB", (target_w, combined_h), (255, 255, 255))
combined.paste(im_apr, (0, 0))                          # first row
combined.paste(space, (0, im_apr.size[1]))
combined.paste(im_oct, (0, im_apr.size[1] + gap))       # second row

combined.save("Sensitivity.png", dpi=(600, 600))
print("Saved: Sensitivity.png")